In [0]:
# 1. Configuración

import time
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType
)

CATALOG = "workspace"
SCHEMA  = "si7006_t2"
VOL_STREAM_IN     = f"/Volumes/{CATALOG}/{SCHEMA}/stream_input"
VOL_CHECKPOINTS   = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"
TBL_BRONZE        = f"{CATALOG}.{SCHEMA}.orders_bronze"
CHECKPOINT_BRONZE = f"{VOL_CHECKPOINTS}/bronze_orders"
SCHEMA_BRONZE     = f"{VOL_CHECKPOINTS}/bronze_orders_schema"

print(f"Stream input: {VOL_STREAM_IN}")
print(f"Tabla:        {TBL_BRONZE}")
print(f"Checkpoint:   {CHECKPOINT_BRONZE}")

Stream input: /Volumes/workspace/si7006_t2/stream_input
Tabla:        workspace.si7006_t2.orders_bronze
Checkpoint:   /Volumes/workspace/si7006_t2/checkpoints/bronze_orders


In [0]:
# 2. Schema explícito de los eventos

schema_eventos = StructType([
    StructField("InvoiceNo",   StringType(),    True),
    StructField("StockCode",   StringType(),    True),
    StructField("Description", StringType(),    True),
    StructField("Quantity",    IntegerType(),   True),
    StructField("InvoiceDate", TimestampType(), True),
    StructField("UnitPrice",   DoubleType(),    True),
    StructField("CustomerID",  IntegerType(),   True),
    StructField("Country",     StringType(),    True),
])

In [0]:
# 3. Lectura streaming con Auto Loader + metadata de auditoría

df_bronze = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", SCHEMA_BRONZE)
        .option("cloudFiles.inferColumnTypes", "false")
        .schema(schema_eventos)
        .load(VOL_STREAM_IN)
        .withColumn("ingested_at",         F.current_timestamp())
        .withColumn("source_file",         F.col("_metadata.file_path"))
        .withColumn("source_file_size",    F.col("_metadata.file_size"))
        .withColumn("source_file_modtime", F.col("_metadata.file_modification_time"))
)

df_bronze.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- ingested_at: timestamp (nullable = false)
 |-- source_file: string (nullable = false)
 |-- source_file_size: long (nullable = false)
 |-- source_file_modtime: timestamp (nullable = false)



In [0]:
# 4. Loop de ingesta con Trigger.AvailableNow
# Serverless Free Edition no soporta triggers infinitos.
# Cada iteración procesa los archivos nuevos disponibles y termina.

ITERACIONES    = 20
PAUSA_SEGUNDOS = 12

for i in range(1, ITERACIONES + 1):
    query = (
        df_bronze.writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", CHECKPOINT_BRONZE)
            .option("mergeSchema", "true")
            .trigger(availableNow=True)
            .toTable(TBL_BRONZE)
    )
    query.awaitTermination()

    total = spark.sql(f"SELECT COUNT(*) AS n FROM {TBL_BRONZE}").collect()[0]["n"]
    print(f"[{i:02d}/{ITERACIONES}] {time.strftime('%H:%M:%S')} | total: {total:,}")

    if i < ITERACIONES:
        time.sleep(PAUSA_SEGUNDOS)

[01/20] 16:45:43 | total: 6,000
[02/20] 16:46:00 | total: 6,000
[03/20] 16:46:16 | total: 6,000
[04/20] 16:46:32 | total: 6,000
[05/20] 16:46:48 | total: 6,000
[06/20] 16:47:05 | total: 6,000
[07/20] 16:47:22 | total: 6,000
[08/20] 16:47:39 | total: 6,000
[09/20] 16:47:56 | total: 6,000
[10/20] 16:48:12 | total: 6,000
[11/20] 16:48:28 | total: 6,000
[12/20] 16:48:44 | total: 6,000
[13/20] 16:49:00 | total: 6,000
[14/20] 16:49:16 | total: 6,000
[15/20] 16:49:32 | total: 6,000
[16/20] 16:49:48 | total: 6,000
[17/20] 16:50:04 | total: 6,000
[18/20] 16:50:20 | total: 6,000
[19/20] 16:50:36 | total: 6,000
[20/20] 16:50:53 | total: 6,000


In [0]:
# 5. Historial Delta y validación

spark.sql(f"DESCRIBE HISTORY {TBL_BRONZE}").select(
    "version", "timestamp", "operation",
    "operationMetrics.numOutputRows"
).show(truncate=False)

spark.sql(f"""
    SELECT COUNT(*) AS total,
           COUNT(DISTINCT source_file) AS archivos,
           COUNT(DISTINCT InvoiceNo)   AS invoices,
           MIN(ingested_at)            AS primer_ingest,
           MAX(ingested_at)            AS ultimo_ingest
    FROM {TBL_BRONZE}
""").show(truncate=False)

spark.sql(f"SELECT * FROM {TBL_BRONZE} ORDER BY ingested_at DESC LIMIT 5").show(truncate=False)

+-------+-------------------+----------------+-------------+
|version|timestamp          |operation       |numOutputRows|
+-------+-------------------+----------------+-------------+
|4      |2026-04-29 16:45:42|STREAMING UPDATE|1500         |
|3      |2026-04-28 16:33:26|STREAMING UPDATE|1500         |
|2      |2026-04-28 16:00:33|STREAMING UPDATE|2500         |
|1      |2026-04-28 16:00:31|STREAMING UPDATE|500          |
|0      |2026-04-28 15:51:25|CREATE TABLE    |NULL         |
+-------+-------------------+----------------+-------------+

+-----+--------+--------+----------------------+-----------------------+
|total|archivos|invoices|primer_ingest         |ultimo_ingest          |
+-----+--------+--------+----------------------+-----------------------+
|6000 |120     |2399    |2026-04-28 16:00:25.34|2026-04-29 16:45:34.073|
+-----+--------+--------+----------------------+-----------------------+

+---------+---------+-----------------------------------+--------+------------------